# Brain MRI tumour research - cloud GPU pipeline (Colab)

Runs **audit -> legacy evaluation -> training (resumable) -> validation -> comparison -> 3D inference test** on a cloud GPU. Nothing heavy runs on your own computer.

**Before you start (one-time, done by you):**
1. `Runtime > Change runtime type > GPU` (T4 is enough; the cells stop if no GPU is present - they never fall back to CPU).
2. `Secrets` (key icon, left bar): add `GITHUB_TOKEN` = a **fine-grained, read-only** token limited to the repository `brain-mri-tumor-3d-research` (GitHub > Settings > Developer settings). Enable *Notebook access*. Never paste a token into a cell or into a chat.
3. Run cells top to bottom. Google will ask **you** to approve Drive access in the mount cell.

Persistent results go to `MyDrive/brain-tumor-3d-print/runs_v2/` (new folder; your old `weights/`, `exports/`, `logs/` are only **read**). A free Colab session can end at any time: **re-run the same training cell** and it resumes from the last epoch.

> Research software, not a medical device. Metrics describe research validation only.

In [ ]:
# 1. GPU check - stops here if there is no usable GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: choose Runtime > Change runtime type > GPU, then reconnect and re-run."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory / 2**30:.1f} GB | compute capability {p.major}.{p.minor} | torch {torch.__version__} / CUDA {torch.version.cuda}")

In [ ]:
# 2. Persistent storage (Google Drive) - Google will ask you to approve access
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/brain-tumor-3d-print')      # the folder used by the original notebook
RUNS = BASE / 'runs_v2'
RUNS.mkdir(parents=True, exist_ok=True)
print('persistent run directory:', RUNS)

In [ ]:
# 3. Code from the PRIVATE repository (token comes from Colab Secrets, is never printed, and is removed from the git config)
import os, subprocess
from google.colab import userdata
REPO = '/content/brain-mri-tumor-3d-research'
URL = 'https://github.com/yamanmohanned/brain-mri-tumor-3d-research.git'
if not os.path.isdir(REPO):
    tok = userdata.get('GITHUB_TOKEN')
    r = subprocess.run(['git', 'clone', '--depth', '1', URL.replace('https://', f'https://x-access-token:{tok}@'), REPO], capture_output=True, text=True)
    del tok
    if r.returncode:
        raise RuntimeError('git clone failed - check that the GITHUB_TOKEN secret exists, has Notebook access and read access to the repo')
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', URL], check=True)
%cd /content/brain-mri-tumor-3d-research
!pip -q install monai nibabel scikit-image trimesh pydicom pyyaml pandas matplotlib kagglehub
!git log -1 --oneline

In [ ]:
# 4. BraTS 2020 -> downloaded INTO the Colab VM (public Kaggle dataset used by the original notebook); layout is verified, not assumed
!python scripts/cloud_run.py get-brats2020 --record {RUNS}/dataset_brats2020.json
import json
DATA = json.load(open(RUNS / 'dataset_brats2020.json'))['root']
print('DATA =', DATA)

In [ ]:
# 5. The ORIGINAL SegResNet checkpoint from Drive: read-only; a hashed backup copy is kept in the run folder
import hashlib, shutil
SRC = BASE / 'weights' / 'segresnet_best.pth'
assert SRC.exists(), f'{SRC} not found - locate the file saved by the original notebook'
LEGACY = RUNS / 'legacy' / 'segresnet_best.pth'
LEGACY.parent.mkdir(exist_ok=True)
if not LEGACY.exists():
    shutil.copy2(SRC, LEGACY)
sha = hashlib.sha256(open(LEGACY, 'rb').read()).hexdigest()
print('legacy checkpoint sha256:', sha, '| size MB:', round(LEGACY.stat().st_size / 2**20, 1))

## Task A - dataset audit (+ check that the reconstructed legacy split matches the original notebook)
Expected: patients 296, 186, 151 have whole-tumour volume **outside** 40-100 cm3 and patient 283 **inside**. A mismatch is reported, never "fixed" by editing data.

In [ ]:
!python scripts/cloud_run.py audit --data-root {DATA} --out {RUNS}/audit_brats2020 --manifest splits/legacy_seed42.json

## Task B - quick evaluation of the original checkpoint (3 validation patients; does not modify the checkpoint or its card)

In [ ]:
!python scripts/cloud_run.py eval --checkpoint {LEGACY} --data-root {DATA} --manifest splits/legacy_seed42.json --split val --limit 3 --out {RUNS}/eval_check --no-card-update

## Task C - full evaluation of the ORIGINAL SegResNet on the legacy test split
**Provenance:** the notebook's `train_model` updates weights only from the training patients and picks the best epoch only on validation patients, and never reads the test split.
But (a) no file hash of the checkpoint was ever recorded, (b) the epoch-3 log of SegResNet and the first 15 epochs of the U-Net are missing from the notebook, and
(c) the notebook compared SegResNet and U-Net on this test split and then called SegResNet the "winner". So **this test split is not an untouched benchmark** and the
checkpoint's provenance is not verified. The evaluation therefore runs *without* `--assert-trained-on-manifest`; the card will say so. Results are kept for reproducibility only.

In [ ]:
RUN_FULL_TEST_EVAL = True      # set False to skip
if RUN_FULL_TEST_EVAL:
    !python scripts/cloud_run.py eval --checkpoint {LEGACY} --data-root {DATA} --manifest splits/legacy_seed42.json --split test --out {RUNS}/eval_segresnet

## Training - retrain with the improved recipe (justification: the original run stopped at 30 epochs while validation Dice was still rising 0.52 -> 0.76)
Resumable and budgeted: if the session ends, **re-run this cell**. `--max-hours 3.5` stops cleanly after the epoch that crosses the budget.
Selection uses the **validation** split only; the test split is never loaded by training.

In [ ]:
RUN_NAME = 'segresnet_dicece_v1'
!python scripts/cloud_run.py train --runs-dir {RUNS} --run-name {RUN_NAME} --data-root {DATA} --manifest splits/legacy_seed42.json --max-hours 3.5 --cache-dir /content/cache

## Compare baseline vs new model on the VALIDATION split under identical conditions (model selection split - not the test set)

In [ ]:
NEW = RUNS / RUN_NAME / 'best.pth'
!python scripts/cloud_run.py eval --checkpoint {LEGACY} --data-root {DATA} --manifest splits/legacy_seed42.json --split val --out {RUNS}/eval_val_legacy --no-card-update
!python scripts/cloud_run.py eval --checkpoint {NEW} --data-root {DATA} --manifest splits/legacy_seed42.json --split val --out {RUNS}/eval_val_new --no-card-update
!python -m braintumor compare --a {RUNS}/eval_val_legacy --b {RUNS}/eval_val_new --out {RUNS}/compare_val

## Final held-out evaluation of the NEW model - run ONCE, only after the model and settings are frozen
The legacy test split is *not* untouched (see Task C), so this gives an internal, comparison-contaminated estimate. For a fresh test set / external cohort build a combined split
(`python -m braintumor split-combined ...`, see USER_GUIDE.md) from additional datasets.

In [ ]:
RUN_FINAL_EVAL_NEW = False     # leave False until the recipe is final
if RUN_FINAL_EVAL_NEW:
    !python scripts/cloud_run.py eval --checkpoint {NEW} --data-root {DATA} --manifest splits/legacy_seed42.json --split test --out {RUNS}/eval_new_test

## 3D inference test (segmentation -> physical measurements -> meshes -> NIfTI/HTML export) on one case, on the GPU

In [ ]:
!python scripts/cloud_run.py infer3d --checkpoint {NEW if NEW.exists() else LEGACY} --data-root {DATA} --out {RUNS}/infer3d_test
!ls -la {RUNS}/infer3d_test

In [ ]:
# Summary of what exists in persistent storage
!find {RUNS} -maxdepth 2 -type f \( -name "*.json" -o -name "*.csv" -o -name "*.pth" -o -name "*.png" \) | sort | head -80